# Danish Housing — Data Cleaning
**Output:** `FinalData/all_data_mun.csv` and `FinalData/all_data_reg.csv`

All municipality names are standardised to the GeoJSON convention:
- `København` / `Copenhagen` → `Københavns`
- `Vesthimmerland` / `Vesthimmerlands` → `Vesthimmerlands`
- `Bornholm` + `Christiansø` → `Bornholms Regionskommune`
- `Nordfyn` → `Nordfyns`

In [37]:
import os, json
import numpy as np
import pandas as pd

# Single source of truth: all raw name variants → final standardised name
MUN_RENAME = {
    'Copenhagen':   'Københavns',
    'København':    'Københavns',
    'Vesthimmerland':  'Vesthimmerlands',
    'Christiansø':  'Bornholms Regionskommune',
    'Bornholm':     'Bornholms Regionskommune',
    'Nordfyn':      'Nordfyns',
}

REGION_MAP = {
    'Capital Region': [
        'Københavns','Frederiksberg','Gentofte','Gladsaxe','Helsingør',
        'Hillerød','Hvidovre','Lyngby-Taarbæk','Rødovre','Ballerup',
        'Brøndby','Dragør','Egedal','Fredensborg','Frederikssund',
        'Furesø','Gribskov','Halsnæs','Herlev','Høje-Taastrup',
        'Hørsholm','Ishøj','Rudersdal','Tårnby','Vallensbæk',
        'Albertslund','Allerød','Glostrup','Bornholms Regionskommune',
    ],
    'Zealand Region': [
        'Roskilde','Holbæk','Næstved','Slagelse','Køge',
        'Guldborgsund','Vordingborg','Kalundborg','Ringsted',
        'Sorø','Lejre','Stevns','Solrød','Faxe','Odsherred','Greve','Lolland',
    ],
    'Central Jutland': [
        'Aarhus','Randers','Viborg','Silkeborg','Herning',
        'Horsens','Skanderborg','Holstebro','Skive',
        'Ringkøbing-Skjern','Favrskov','Hedensted',
        'Syddjurs','Ikast-Brande','Norddjurs','Struer',
        'Odder','Lemvig','Samsø',
    ],
    'North Jutland': [
        'Aalborg','Hjørring','Frederikshavn','Thisted',
        'Mariagerfjord','Jammerbugt','Vesthimmerlands',
        'Brønderslev','Rebild','Morsø','Læsø',
    ],
    'Southern Denmark': [
        'Odense','Esbjerg','Vejle','Kolding','Sønderborg',
        'Aabenraa','Svendborg','Haderslev','Faaborg-Midtfyn',
        'Varde','Fredericia','Vejen','Assens','Tønder',
        'Middelfart','Nyborg','Nordfyns','Billund',
        'Kerteminde','Langeland','Ærø','Fanø',
    ],
}

MUN_TO_REGION = {
    mun: region
    for region, muns in REGION_MAP.items()
    for mun in muns
}

print('Config ready. Municipalities in lookup:', len(MUN_TO_REGION))

Config ready. Municipalities in lookup: 98


## 1. Housing transactions

In [38]:
with open('zip_to_muni_clean.json') as f:
    zip_to_muni = {int(k): v for k, v in json.load(f).items()}

# Standardise names already in the zip map
zip_to_muni = {k: MUN_RENAME.get(v, v) for k, v in zip_to_muni.items()}

# Manual corrections for mis-mapped zip codes
zip_to_muni.update({
    2610: 'Rødovre',
    2650: 'Hvidovre',
    2730: 'Herlev',
    2740: 'Glostrup',
    2760: 'Glostrup',
    2770: 'Tårnby',
    6330: 'Aabenraa',
    9500: 'Mariagerfjord',
    9550: 'Mariagerfjord',
    9560: 'Mariagerfjord',
    9574: 'Mariagerfjord',
    9580: 'Mariagerfjord',
    9592: 'Mariagerfjord',
    4180: 'Sorø',
    4190: 'Sorø',
})

df_housing = pd.read_parquet('DKHousingPrices.parquet', engine='fastparquet')
df_housing['zip_code'] = df_housing['zip_code'].astype(int)
df_housing['year'] = pd.to_datetime(df_housing['date']).dt.year
df_housing = df_housing[df_housing['year'].between(2008, 2024)]

df_housing['Municipality'] = df_housing['zip_code'].map(zip_to_muni)
df_housing['Municipality'] = df_housing['Municipality'].replace(MUN_RENAME)
df_housing['Region'] = df_housing['Municipality'].map(MUN_TO_REGION)

print(f'Missing municipality: {df_housing["Municipality"].isna().sum()}')
print(f'Missing region:       {df_housing["Region"].isna().sum()}')
print(f'Total rows:           {len(df_housing)}')

Missing municipality: 0
Missing region:       0
Total rows:           1028034


In [39]:
agg_dict = {
    'purchase_price':                      ['mean','median'],
    'sqm_price':                           ['mean','median'],
    'sqm':                                 'mean',
    'no_rooms':                            'mean',
    '%_change_between_offer_and_purchase': 'mean',
    'nom_interest_rate%':                  'mean',
    'dk_ann_infl_rate%':                   'mean',
    'yield_on_mortgage_credit_bonds%':     'mean',
    'house_id':                            'count',
}

df_housing_mun = df_housing.groupby(['Municipality','Region','year']).agg(agg_dict).reset_index()
df_housing_mun.columns = ['_'.join(c).strip('_') for c in df_housing_mun.columns]
df_housing_mun = df_housing_mun.rename(columns={'house_id_count':'no_sales'})

df_housing_reg = df_housing.groupby(['Region','year']).agg(agg_dict).reset_index()
df_housing_reg.columns = ['_'.join(c).strip('_') for c in df_housing_reg.columns]
df_housing_reg = df_housing_reg.rename(columns={'house_id_count':'no_sales'})

print('Housing mun:', df_housing_mun.shape)
print('Housing reg:', df_housing_reg.shape)

Housing mun: (1666, 14)
Housing reg: (85, 13)


## 2. Income data

In [40]:
families = pd.read_csv('families.csv', sep=';')
couples  = pd.read_csv('couples.csv',  sep=';')
singles  = pd.read_csv('Singles.csv',  sep=';')

combined = pd.concat([families, couples, singles], ignore_index=True)
combined = combined.rename(columns={'Unnamed: 0': 'Municipality'})
income = combined.groupby('Municipality', as_index=False).mean(numeric_only=True)
income = income[income['Municipality'] != 'All Denmark']

# Standardise names
income['Municipality'] = income['Municipality'].replace(MUN_RENAME)

year_cols = [c for c in income.columns if str(c).isdigit()]
income_long = income.melt(id_vars=['Municipality'], value_vars=year_cols,
                          var_name='year', value_name='income')
income_long['year'] = income_long['year'].astype(int)
income_long = income_long[income_long['year'].between(2008, 2024)]
income_long['Region'] = income_long['Municipality'].map(MUN_TO_REGION)

missing = income_long[income_long['Region'].isna()]['Municipality'].unique()
print('Income — municipalities without region:', missing)
print('Income long shape:', income_long.shape)

Income — municipalities without region: <ArrowStringArray>
[]
Length: 0, dtype: str
Income long shape: (1666, 4)


## 3. Population data

The key fix: `'København'` is renamed to `'Københavns'` via `MUN_RENAME` **before** any merge, so it matches the housing and income tables.

In [41]:
raw = pd.read_excel('Population_by_area.xlsx', header=None)

# Row 0 = quarter labels, column 0 = municipality names
quarter_labels = raw.iloc[0, 1:].tolist()   # ['2008K1', '2008K2', ...]
muni_names     = raw.iloc[1:, 0].tolist()   # ['Hele landet', 'København', ...]
values         = raw.iloc[1:, 1:].values    # numeric population matrix

df_pop_raw = pd.DataFrame(values, columns=quarter_labels)
df_pop_raw.insert(0, 'Municipality', muni_names)

# Filter out national total and regions
exclude_rows = [
    'Hele landet',
    'Region Hovedstaden', 'Region Sjælland', 'Region Syddanmark',
    'Region Midtjylland', 'Region Nordjylland',
]
df_pop_raw = df_pop_raw[
    df_pop_raw['Municipality'].notna() &
    ~df_pop_raw['Municipality'].isin(exclude_rows)
].copy()

# Standardise names BEFORE anything else
df_pop_raw['Municipality'] = df_pop_raw['Municipality'].replace(MUN_RENAME)

# Melt to long format
df_pop_long = df_pop_raw.melt(
    id_vars=['Municipality'],
    var_name='quarter_str',
    value_name='population'
)
df_pop_long['population'] = pd.to_numeric(df_pop_long['population'], errors='coerce')

# Extract year from 'K' format: '2008K1' → 2008
df_pop_long['year'] = df_pop_long['quarter_str'].str[:4].astype(int)
df_pop_long = df_pop_long[df_pop_long['year'].between(2008, 2024)]

# Average 4 quarters → yearly mean, then sum Bornholm+Christiansø
df_pop_yearly = (
    df_pop_long
    .groupby(['Municipality', 'year'], as_index=False)['population']
    .mean()
    .groupby(['Municipality', 'year'], as_index=False)['population']
    .sum()
)

df_pop_yearly['Region'] = df_pop_yearly['Municipality'].map(MUN_TO_REGION)

# Verify
print('Shape:', df_pop_yearly.shape)
print('NaN in population:', df_pop_yearly['population'].isna().sum())
check = df_pop_yearly[df_pop_yearly['Municipality'].isin(['Københavns', 'Vesthimmerlands'])]
print(check.groupby('Municipality')['population'].agg(['count', 'mean', 'min', 'max']))

Shape: (1666, 4)
NaN in population: 0
                 count           mean       min       max
Municipality                                             
Københavns          17  592379.808824  512558.5  662770.5
Vesthimmerlands     17   37233.176471   35920.5   38421.5


## 4. Merge

In [42]:
# ── Municipality level ───────────────────────────────────────────────────────
all_data_mun = (
    df_housing_mun
    .merge(income_long[['Municipality','year','income']], on=['Municipality','year'], how='left')
    .merge(df_pop_yearly[['Municipality','year','population']], on=['Municipality','year'], how='left')
)

pop_null = all_data_mun[all_data_mun['population'].isna()]['Municipality'].unique()
inc_null = all_data_mun[all_data_mun['income'].isna()]['Municipality'].unique()
print('Missing population:', pop_null)
print('Missing income:    ', inc_null)

check = all_data_mun[all_data_mun['Municipality'].isin(['Københavns','Vesthimmerlands'])]
print('\nVerification — should have no NaN in population:')
print(check[['Municipality','year','income','population']].head(6))

# ── Region level ─────────────────────────────────────────────────────────────
income_reg = income_long.groupby(['Region','year'], as_index=False)['income'].mean()
pop_reg    = df_pop_yearly.groupby(['Region','year'], as_index=False)['population'].sum()

all_data_reg = (
    df_housing_reg
    .merge(income_reg, on=['Region','year'], how='left')
    .merge(pop_reg,   on=['Region','year'], how='left')
)



Missing population: <ArrowStringArray>
[]
Length: 0, dtype: str
Missing income:     <ArrowStringArray>
[]
Length: 0, dtype: str

Verification — should have no NaN in population:
    Municipality  year         income  population
850   Københavns  2008  259291.333333   512558.50
851   Københavns  2009  266756.333333   521887.00
852   Københavns  2010  287346.333333   532085.00
853   Københavns  2011  295723.000000   542883.25
854   Københavns  2012  302645.333333   552612.50
855   Københavns  2013  311350.666667   563268.00


## 4. Save file

In [43]:
# ── Save ─────────────────────────────────────────────────────────────────────
all_data_mun.to_csv('../FinalData/all_data_mun.csv', index=False)
all_data_reg.to_csv('../FinalData/all_data_reg.csv', index=False)

print('\nSaved:')
print('  all_data_mun:', all_data_mun.shape)
print('  all_data_reg:', all_data_reg.shape)


Saved:
  all_data_mun: (1666, 16)
  all_data_reg: (85, 15)
